# SpeciesNet, final

Notebook 15's first run worked and produced the largest jump in this project. It also had two
defects, both fixed here.

**It selected the winning model by looking at `ood_val`.** The last cell picked whichever
candidate scored best on the official split — the exact thing fourteen notebooks were built to
avoid. Every candidate is now scored on the **36 held-out cameras**, the winner chosen there,
and `ood_val` read only afterwards.

**The `empty` class was a dumping ground** — precision 0.369, recall 0.998, with 1,024 animal
images called empty. Section 6 measures why, and tests a candidate fix *on the selection
cameras* rather than assuming it helps. The first attempt at that fix made things worse, which
is recorded here rather than quietly dropped.

### What this notebook is

The pipeline unchanged: MegaDetector boxes from notebook 11 -> crop -> **SpeciesNet** ->
classifier -> burst vote. SpeciesNet is an EfficientNetV2-M trained by Google on 65 million
camera-trap images, and its own documented architecture is exactly this one.

Nothing expensive re-runs. The SpeciesNet pass is cached in `speciesnet_*.json`, so this
notebook is minutes, not an hour.

### The caveat that belongs in the report

SpeciesNet was trained on "curated images from the Wildlife Insights community and
publicly-available repositories". **iWildCam is publicly available on LILA.** So SpeciesNet may
have trained on these very images. That does not reduce its practical value — for a real
deployment it is simply the right tool — but it means a score here is not a clean
generalization measurement the way notebook 11's 0.598 was. The cameras are unseen to *our*
classifier; they may not be unseen to SpeciesNet.

## 1. Setup

In [1]:
import io
import json
import time
import shutil
import zipfile
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

plt.rcParams["figure.figsize"] = (11, 4)
print("torch:", torch.__version__, "| device:", DEVICE)

torch: 2.11.0+cu130 | device: cuda


## 2. Configuration

In [2]:
from google.colab import drive
drive.mount("/content/drive")

ZIP_PATH = Path("/content/drive/MyDrive/IWildCam v2.0 2020 WILDS Dataset.zip")
CACHE_DIR = Path("/content/drive/MyDrive/iwildcam_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

IMG_SIZE = 192
N_CLASSES = 10
CAP_TRAIN = 3000
CAP_EVAL = 600
HOLDOUT_FRAC = 0.15

MODEL_NAME = None       # None = let speciesnet resolve its own default weights
CROP_SIZE = 480
MIN_CONF = 0.15
WORK_MAX = 1333
CHUNK = 400
TOP_K = 10
SEQ_RULE = "max"        # notebook 12 measured +0.041 honest over "mean"

Mounted at /content/drive


## 3. Data

The same ten classes, sampling, splits and seed as notebooks 06-14, so every number here sits
in the same table as notebook 11's 0.598.

In [3]:
ROOT = "iwildcam_v2.0"


class IWildCamSource:
    "Reads from the WILDS ZIP, and can reopen after a Drive mount drops."

    def __init__(self, path):
        self.path = Path(path)
        self.is_zip = self.path.suffix.lower() == ".zip"
        self._open()

    def _open(self):
        if self.is_zip:
            self.zf = zipfile.ZipFile(self.path)
        else:
            self.dir = self.path if self.path.name == ROOT else self.path / ROOT

    def reopen(self):
        try:
            self.zf.close()
        except Exception:
            pass
        self._open()

    def read(self, rel):
        if self.is_zip:
            return self.zf.read(f"{ROOT}/{rel}")
        return (self.dir / rel).read_bytes()


source = IWildCamSource(ZIP_PATH)
metadata = pd.read_csv(io.BytesIO(source.read("metadata.csv")), index_col=0)
categories = pd.read_csv(io.BytesIO(source.read("categories.csv")))

COMMON = {
    "empty": "empty (false trigger)", "meleagris ocellata": "ocellated turkey",
    "crax rubra": "great curassow", "tayassu pecari": "white-lipped peccary",
    "cephalophus nigrifrons": "black-fronted duiker", "dasyprocta punctata": "agouti",
    "leopardus pardalis": "ocelot", "mazama temama": "red brocket deer",
    "puma concolor": "puma", "urocyon cinereoargenteus": "gray fox",
}
scientific = dict(zip(categories.y, categories.name))
top_labels = metadata[metadata.split == "train"].y.value_counts().head(N_CLASSES).index.tolist()
CLASS_NAMES = [COMMON.get(scientific[y], scientific[y]) for y in top_labels]
SCIENTIFIC_NAMES = [scientific[y] for y in top_labels]
label_to_index = {y: i for i, y in enumerate(top_labels)}
EMPTY_INDEX = CLASS_NAMES.index("empty (false trigger)")
ANIMAL_INDICES = [i for i in range(N_CLASSES) if i != EMPTY_INDEX]


def sample_split(split, cap, seed=SEED):
    rows = metadata[(metadata.split == split) & (metadata.y.isin(top_labels))]
    rng = np.random.default_rng(seed)
    kept = []
    for y in top_labels:
        cls = rows[rows.y == y]
        groups = list(cls.groupby("seq_id").indices.values())
        rng.shuffle(groups)
        order, depth = [], 0
        while len(order) < len(cls):
            taken = [cls.index[g[depth]] for g in groups if depth < len(g)]
            if not taken:
                break
            order.extend(taken)
            depth += 1
        kept.extend(order[:cap])
    return metadata.loc[kept]


SPLITS = {"train": sample_split("train", CAP_TRAIN),
          "id_test": sample_split("id_test", CAP_EVAL),
          "ood_val": sample_split("val", CAP_EVAL)}
LABELS = {n: np.array([label_to_index[v] for v in f.y], dtype=np.int64)
          for n, f in SPLITS.items()}


def camera_holdout(frame, target_frac=HOLDOUT_FRAC, seed=SEED):
    rng = np.random.default_rng(seed)
    sizes = frame.location_remapped.value_counts()
    held, n = [], 0
    for cam in rng.permutation(sizes.index.to_numpy()):
        if n >= target_frac * len(frame):
            break
        held.append(cam)
        n += sizes[cam]
    for y in frame.y.unique():
        if (frame[frame.location_remapped.isin(held)].y == y).sum() == 0:
            held.append(frame[(frame.y == y)].location_remapped.value_counts().index[0])
        if (frame[~frame.location_remapped.isin(held)].y == y).sum() == 0:
            held = [c for c in held
                    if (frame[frame.location_remapped == c].y == y).mean() < 1.0]
    mask = frame.location_remapped.isin(held).to_numpy()
    return ~mask, mask


train_frame = SPLITS["train"]
FIT_MASK, SEL_MASK = camera_holdout(train_frame)
y_fit, y_sel = LABELS["train"][FIT_MASK], LABELS["train"][SEL_MASK]
SEL_FRAME = train_frame[SEL_MASK]

SITE_SEPARABLE = ["black-fronted duiker", "empty (false trigger)"]


def honest_macro_f1(truth, preds, exclude=SITE_SEPARABLE):
    per_class = f1_score(truth, preds, average=None, labels=range(N_CLASSES), zero_division=0)
    return per_class[[i for i, n in enumerate(CLASS_NAMES) if n not in exclude]].mean()


def aggregate(probs, frame, how=SEQ_RULE):
    "One answer per burst: the mean of the probabilities, or the single strongest frame."
    out = np.empty(len(frame), dtype=int)
    seq = frame.seq_id.to_numpy()
    for s in np.unique(seq):
        rows = np.flatnonzero(seq == s)
        out[rows] = (probs[rows].mean(axis=0).argmax() if how == "mean"
                     else np.unravel_index(probs[rows].argmax(), probs[rows].shape)[1])
    return out


print(f"{'split':>9} {'images':>8} {'cameras':>9}")
for name, frame in SPLITS.items():
    print(f"{name:>9} {len(frame):>8,} {frame.location_remapped.nunique():>9}")
print(f"\nfit {FIT_MASK.sum():,} / selection {SEL_MASK.sum():,} images, "
      f"{SEL_FRAME.location_remapped.nunique()} selection cameras")

    split   images   cameras
    train   29,740       208
  id_test    3,420       137
  ood_val    4,361        29

fit 25,211 / selection 4,529 images, 36 selection cameras


## 4. Boxes and crops, from notebook 11

No detector runs. Notebook 11 wrote one padded box and its confidence per image.

In [4]:
BOXES, CONF = {}, {}
missing = []
for name in SPLITS:
    path = CACHE_DIR / f"det_megadetector_bioclip_{name}_boxes.npy"
    try:
        arr = np.load(path)
        if len(arr) != len(SPLITS[name]):
            raise ValueError(f"{len(arr)} rows, expected {len(SPLITS[name])}")
        BOXES[name], CONF[name] = arr[:, :4], arr[:, 4]
    except Exception as exc:
        missing.append(f"{path.name}: {type(exc).__name__}: {exc}")

if missing:
    print("could not read notebook 11's boxes:")
    for line in missing:
        print("  ", line)
    print("\nfalling back to whole frames - run notebook 11 first for the real comparison")
    for name in SPLITS:
        BOXES[name] = np.tile(np.array([0, 0, 1, 1], np.float32), (len(SPLITS[name]), 1))
        CONF[name] = np.zeros(len(SPLITS[name]), np.float32)
else:
    for name in SPLITS:
        side = np.maximum(BOXES[name][:, 2] - BOXES[name][:, 0],
                          BOXES[name][:, 3] - BOXES[name][:, 1])
        print(f"{name:>9}: median box side {np.median(side):.2f}, "
              f"{(CONF[name] < MIN_CONF).mean():.1%} below the confidence floor "
              f"(those get the whole frame)")


def read_original(rel, attempts=4):
    for k in range(attempts):
        try:
            return source.read(rel)
        except (OSError, zipfile.BadZipFile, KeyError):
            if k == attempts - 1:
                return None
            try:
                drive.mount("/content/drive", force_remount=True)
            except Exception:
                pass
            time.sleep(2 ** k)
            source.reopen()
    return None


def crop_for(split, i, size=CROP_SIZE):
    raw = read_original(f"train/{SPLITS[split].filename.iloc[i]}")
    if raw is None:
        return None
    try:
        img = Image.open(io.BytesIO(raw)).convert("RGB")
    except Exception:
        return None
    if max(img.size) > WORK_MAX:
        scale = WORK_MAX / max(img.size)
        img = img.resize((max(int(img.width * scale), 1), max(int(img.height * scale), 1)),
                         Image.BILINEAR)
    arr = np.asarray(img)
    box = (BOXES[split][i] if CONF[split][i] >= MIN_CONF
           else np.array([0, 0, 1, 1], np.float32))
    h, w = arr.shape[:2]
    x0, y0, x1, y1 = box
    x0, x1 = int(x0 * w), int(np.ceil(x1 * w))
    y0, y1 = int(y0 * h), int(np.ceil(y1 * h))
    x0, y0 = max(x0, 0), max(y0, 0)
    x1, y1 = min(max(x1, x0 + 2), w), min(max(y1, y0 + 2), h)
    return np.asarray(Image.fromarray(arr[y0:y1, x0:x1]).resize((size, size), Image.BICUBIC))

    train: median box side 0.50, 18.2% below the confidence floor (those get the whole frame)
  id_test: median box side 0.49, 21.3% below the confidence floor (those get the whole frame)
  ood_val: median box side 0.50, 17.7% below the confidence floor (those get the whole frame)


## 5. SpeciesNet predictions

Read from `speciesnet_*.json` if the pass has already run. Otherwise the classifier is loaded
and the pass runs once, checkpointed.

The loader discovers the model name rather than hard-coding one, copies the weights somewhere
writable — Colab mounts Kaggle models read-only, and SpeciesNet writes next to them — and
reaches the **classifier sub-model**, because the full ensemble expects a detector we are not
using.

In [5]:
def cache_complete():
    "True when every split already has a full set of SpeciesNet records on Drive."
    for name in SPLITS:
        store = CACHE_DIR / f"speciesnet_{name}.json"
        if not store.exists():
            return False
        try:
            if len(json.loads(store.read_text())) != len(SPLITS[name]):
                return False
        except Exception:
            return False
    return True


def load_classifier():
    "Return (description, runner). Discovers the model name; proves it on one crop."
    try:
        import speciesnet
    except ModuleNotFoundError:
        print("installing speciesnet ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "speciesnet"], check=True)
        import speciesnet

    def weights_copy(handle="google/speciesnet/pyTorch/v4.0.3a/1"):
        try:
            import kagglehub
            src = Path(kagglehub.model_download(handle))
        except Exception as exc:
            print(f"  kagglehub: {type(exc).__name__}: {exc}")
            return None
        dst = Path("/content/speciesnet_weights")
        if not (dst.exists() and any(dst.iterdir())):
            shutil.copytree(src, dst, dirs_exist_ok=True)
        return str(dst)

    constants = [getattr(speciesnet, n) for n in dir(speciesnet)
                 if isinstance(getattr(speciesnet, n, None), str)
                 and "kaggle:" in str(getattr(speciesnet, n)).lower()]
    names, seen = [], set()
    for name in ([MODEL_NAME] if MODEL_NAME else []) + [weights_copy()] + constants:
        if name and name not in seen:
            seen.add(name)
            names.append(name)

    errors = []
    for name in names:
        for cls_name in ("SpeciesNetClassifier", "SpeciesNet"):
            cls = getattr(speciesnet, cls_name, None)
            if cls is None:
                continue
            try:
                model = (cls(name) if cls_name == "SpeciesNetClassifier"
                         else cls(name, components="classifier"))
            except Exception as exc:
                errors.append(f"{cls_name}/{Path(str(name)).name}: "
                              f"{type(exc).__name__}: {str(exc).splitlines()[0][:70]}")
                continue

            # the ensemble object wants a detector; the classifier sub-model does not
            target = next((getattr(model, a) for a in ("classifier", "_classifier")
                           if hasattr(getattr(model, a, None), "predict")), model)

            def runner(crops, _t=target):
                out = []
                for k, crop in enumerate(crops):
                    fname = f"crop_{k}.jpg"
                    prepared = None
                    if hasattr(_t, "preprocess"):
                        for attempt in (lambda: _t.preprocess(crop),
                                        lambda: _t.preprocess(Image.fromarray(crop))):
                            try:
                                prepared = attempt()
                                break
                            except Exception:
                                continue
                    payload = prepared if prepared is not None else crop
                    for call in (lambda: _t.predict(fname, payload),
                                 lambda: _t.predict(filepath=fname, img=payload),
                                 lambda: _t.predict(instances_dict={
                                     "instances": [{"filepath": fname, "image": crop}]})):
                        try:
                            out.append(call())
                            break
                        except TypeError:
                            continue
                    else:
                        raise RuntimeError("no predict signature accepted a crop")
                return out

            try:
                if not normalize_result(runner(
                        [np.zeros((CROP_SIZE, CROP_SIZE, 3), np.uint8)])[0]):
                    raise RuntimeError("returned no usable scores")
            except Exception as exc:
                errors.append(f"{cls_name}/{Path(str(name)).name}: predict failed: "
                              f"{type(exc).__name__}: {str(exc).splitlines()[0][:70]}")
                continue
            return f"{cls_name} / {Path(str(name)).name}", runner

    raise RuntimeError("no SpeciesNet entry point worked:\n  " + "\n  ".join(errors))


def normalize_result(result):
    "Pull {label: score} out of whatever shape the package returned."
    if isinstance(result, dict):
        for key in ("predictions", "classifications"):
            if key in result:
                return normalize_result(result[key])
        if "classes" in result and "scores" in result:
            return dict(zip(result["classes"], result["scores"]))
        if all(isinstance(v, (int, float)) for v in result.values()):
            return dict(result)
    if isinstance(result, (list, tuple)) and result:
        return normalize_result(result[0])
    return {}


def classify_split(split, runner):
    store = CACHE_DIR / f"speciesnet_{split}.json"
    state = CACHE_DIR / f"speciesnet_{split}_state.json"
    n = len(SPLITS[split])
    records = json.loads(store.read_text()) if store.exists() else []
    done = json.loads(state.read_text()).get("done", 0) if state.exists() else 0
    if done >= n and len(records) == n:
        print(f"  {split}: {n:,} already done")
        return records

    records, start = records[:done], time.perf_counter()
    while done < n:
        hi = min(done + CHUNK, n)
        crops = []
        for i in range(done, hi):
            crop = crop_for(split, i)
            crops.append(crop if crop is not None
                         else np.zeros((CROP_SIZE, CROP_SIZE, 3), np.uint8))
        for result in runner(crops):
            scores = normalize_result(result)
            best = sorted(scores.items(), key=lambda kv: -kv[1])[:TOP_K]
            records.append({"labels": [str(l) for l, _ in best],
                            "scores": [float(s) for _, s in best]})
        done = hi
        store.write_text(json.dumps(records))
        state.write_text(json.dumps({"done": done}))
        print(f"  {split}: {done:,}/{n:,} ({time.perf_counter() - start:.0f}s)")
    return records


if cache_complete():
    RECORDS = {n: json.loads((CACHE_DIR / f"speciesnet_{n}.json").read_text())
               for n in SPLITS}
    print("SpeciesNet predictions read from cache - no model loaded, no images read")
else:
    ENTRY, runner = load_classifier()
    print("entry point:", ENTRY)
    RECORDS = {n: classify_split(n, runner) for n in SPLITS}

for name in SPLITS:
    print(f"{name:>9}: {len(RECORDS[name]):,} records")

SpeciesNet predictions read from cache - no model loaded, no images read
    train: 29,740 records
  id_test: 3,420 records
  ood_val: 4,361 records


## 6. Mapping 2000 labels onto ten, and the `empty` problem

SpeciesNet labels are `uuid;class;order;family;genus;species;common name`. Matching goes by
scientific name, then genus, then common name; `blank` becomes our `empty`; anything else is
left unmapped rather than guessed.

The first run put **1,024 animal images into `empty`** — precision 0.369 against recall 0.998.
Two candidate causes, and the cell below separates them.

1. **A mapping artifact.** An image whose labels all fall outside our ten gets an all-zero
   score row, and `np.argmax` of zeros returns 0, which is `empty`. So "I am sure this is a
   squirrel" silently became "this is empty".
2. **SpeciesNet genuinely saying `blank`.** About 18% of crops fell back to whole frames
   because MegaDetector had no confident box, and a whole frame containing a small animal does
   look blank to a *crop* classifier.

Cause 1 has an obvious fix — spread unmapped mass across the nine animal classes, since an
unmapped species is evidence *against* blank. When that was tried it made zero-shot burst
honest F1 **worse**, 0.825 to 0.776: a truly empty frame then becomes a false positive for
some animal, and those cost precision on the eight classes the honest metric counts.

So both variants are kept and the choice is made in section 8 on the held-out cameras. The
measurement below says which cause dominates.

In [6]:
def parse_label(label):
    parts = [p.strip().lower() for p in str(label).split(";")]
    if len(parts) < 7:
        return " ".join(parts), parts[0] if parts else "", parts[-1] if parts else ""
    return f"{parts[4]} {parts[5]}".strip(), parts[4], parts[6]


SCI_TO_INDEX = {s.lower(): i for i, s in enumerate(SCIENTIFIC_NAMES)}
GENUS_TO_INDEX = {}
for i, s in enumerate(SCIENTIFIC_NAMES):
    GENUS_TO_INDEX.setdefault(s.lower().split()[0], []).append(i)
GENUS_TO_INDEX = {g: v[0] for g, v in GENUS_TO_INDEX.items() if len(v) == 1}
COMMON_TO_INDEX = {n.lower(): i for i, n in enumerate(CLASS_NAMES)}


def label_to_class(label):
    "Our class index for a SpeciesNet label, or None when it names something else."
    sci, genus, common = parse_label(label)
    if "blank" in str(label).lower() or "empty" in str(label).lower():
        return EMPTY_INDEX
    if sci in SCI_TO_INDEX:
        return SCI_TO_INDEX[sci]
    if genus in GENUS_TO_INDEX:
        return GENUS_TO_INDEX[genus]
    for name, i in COMMON_TO_INDEX.items():
        if name != "empty (false trigger)" and name in common:
            return i
    return None


def top_label(rec):
    return rec["labels"][int(np.argmax(rec["scores"]))] if rec["labels"] else None


recs = RECORDS["ood_val"]
truth_ood = LABELS["ood_val"]
no_mapped = np.array([all(label_to_class(l) is None for l in r["labels"]) for r in recs])
says_blank = np.array([label_to_class(top_label(r)) == EMPTY_INDEX if r["labels"] else False
                       for r in recs])
really_empty = truth_ood == EMPTY_INDEX

print(f"{'images that are truly empty':<44} {really_empty.mean():>7.1%}")
print(f"{'images whose top label is blank':<44} {says_blank.mean():>7.1%}")
print(f"{'images with NO label mapping to our ten':<44} {no_mapped.mean():>7.1%}   <- cause 1")
print(f"{'   ...of those, actually empty':<44} "
      f"{really_empty[no_mapped].mean() if no_mapped.any() else 0:>7.1%}")
print(f"{'animals whose top label is blank':<44} "
      f"{says_blank[~really_empty].mean():>7.1%}   <- cause 2")
print("\nwhichever share is larger is the one that matters; the first run showed")
print("empty precision 0.369 with recall 0.998, i.e. 1,024 animals called empty")

images that are truly empty                    13.8%
images whose top label is blank                38.1%
images with NO label mapping to our ten         0.3%   <- cause 1
   ...of those, actually empty                 15.4%
animals whose top label is blank               28.4%   <- cause 2

whichever share is larger is the one that matters; the first run showed
empty precision 0.369 with recall 0.998, i.e. 1,024 animals called empty


In [7]:
def score_matrix_plain(records):
    "Mapped score mass only. An image with nothing mapped scores zero everywhere."
    out = np.zeros((len(records), N_CLASSES), np.float32)
    for i, rec in enumerate(records):
        for label, score in zip(rec["labels"], rec["scores"]):
            target = label_to_class(label)
            if target is not None:
                out[i, target] += score
    return out


def score_matrix_spread(records):
    "As above, but unmapped mass is spread over the animal classes instead of vanishing."
    out = np.zeros((len(records), N_CLASSES), np.float32)
    for i, rec in enumerate(records):
        unmapped = 0.0
        for label, score in zip(rec["labels"], rec["scores"]):
            target = label_to_class(label)
            if target is None:
                unmapped += score
            else:
                out[i, target] += score
        if unmapped > 0:
            out[i, ANIMAL_INDICES] += unmapped / len(ANIMAL_INDICES)
        if out[i].sum() == 0:
            out[i, ANIMAL_INDICES] = 1.0 / len(ANIMAL_INDICES)
    return out


VARIANTS = {"plain": score_matrix_plain, "spread unmapped": score_matrix_spread}
VOCAB = sorted({l for name in SPLITS for rec in RECORDS[name] for l in rec["labels"]})
VOCAB_INDEX = {l: i for i, l in enumerate(VOCAB)}
print(f"{len(VOCAB)} distinct SpeciesNet labels across all splits")


def sparse_features(records):
    out = np.zeros((len(records), len(VOCAB)), np.float32)
    for i, rec in enumerate(records):
        for label, score in zip(rec["labels"], rec["scores"]):
            out[i, VOCAB_INDEX[label]] = score
    return out


SPARSE = {n: sparse_features(RECORDS[n]) for n in SPLITS}
print(f"sparse feature block: {SPARSE['train'].shape}")

461 distinct SpeciesNet labels across all splits
sparse feature block: (29740, 461)


## 7. The models

A linear-plus-MLP head on the SpeciesNet score vector, the same head on notebook 11's BioCLIP
crop embeddings, SpeciesNet's own zero-shot answer, and averages of those. Every one is built
for both score-matrix variants, so section 8 chooses the variant and the model together.

In [8]:
class MLPHead(nn.Module):
    def __init__(self, dim, n_classes, hidden=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, hidden), nn.ReLU(),
                                 nn.Dropout(0.2), nn.Linear(hidden, n_classes))

    def forward(self, x):
        return self.net(x)


def train_head(X, y, epochs=40, seed=SEED):
    torch.manual_seed(seed)
    Xt = torch.tensor(X, dtype=torch.float32, device=DEVICE)
    Yt = torch.tensor(y, device=DEVICE)
    weights = torch.tensor(len(y) / (N_CLASSES * np.bincount(y, minlength=N_CLASSES).clip(1)),
                           dtype=torch.float32, device=DEVICE)
    model = MLPHead(X.shape[1], N_CLASSES).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    loss_fn = nn.CrossEntropyLoss(weight=weights)
    for _ in range(epochs):
        model.train()
        order = torch.randperm(len(Xt), device=DEVICE)
        for i in range(0, len(order), 512):
            idx = order[i:i + 512]
            if len(idx) < 2:
                continue
            optimizer.zero_grad()
            loss_fn(model(Xt[idx]), Yt[idx]).backward()
            optimizer.step()
    return model


@torch.no_grad()
def head_probs(model, X):
    model.eval()
    out = []
    for i in range(0, len(X), 4096):
        xb = torch.tensor(X[i:i + 4096], dtype=torch.float32, device=DEVICE)
        out.append(F.softmax(model(xb), dim=1).cpu().numpy())
    return np.concatenate(out)


BIOCLIP = {}
try:
    for name in SPLITS:
        arr = np.load(CACHE_DIR / f"det_megadetector_bioclip_{name}.npy")
        if len(arr) != len(SPLITS[name]):
            raise ValueError("row count mismatch")
        BIOCLIP[name] = arr / np.clip(np.linalg.norm(arr, axis=1, keepdims=True), 1e-9, None)
    BIO_HEAD = train_head(BIOCLIP["train"][FIT_MASK], y_fit)
    print("BioCLIP crop embeddings loaded from notebook 11")
except Exception as exc:
    BIOCLIP, BIO_HEAD = None, None
    print(f"BioCLIP cache unavailable ({type(exc).__name__}: {exc}); SpeciesNet only")


def build_candidates(variant_name):
    "Probability functions for one score-matrix variant, keyed by model name."
    zero = {n: VARIANTS[variant_name](RECORDS[n]) for n in SPLITS}
    feats = {n: np.hstack([SPARSE[n], zero[n]]) for n in SPLITS}
    head = train_head(feats["train"][FIT_MASK], y_fit)

    def norm(m):
        return m / np.clip(m.sum(axis=1, keepdims=True), 1e-9, None)

    out = {"zero shot": lambda s, m=None: norm(zero[s] if m is None else zero[s][m]),
           "probe on SpeciesNet": lambda s, m=None: head_probs(
               head, feats[s] if m is None else feats[s][m])}
    if BIOCLIP is not None:
        out["probe on BioCLIP"] = lambda s, m=None: head_probs(
            BIO_HEAD, BIOCLIP[s] if m is None else BIOCLIP[s][m])
        out["both probes"] = lambda s, m=None: (out["probe on SpeciesNet"](s, m)
                                                + out["probe on BioCLIP"](s, m)) / 2
        out["everything"] = lambda s, m=None: (out["probe on SpeciesNet"](s, m)
                                               + out["probe on BioCLIP"](s, m)
                                               + out["zero shot"](s, m)) / 3
    return out


CANDIDATES = {f"{model}  [{variant}]": fn
              for variant in VARIANTS
              for model, fn in build_candidates(variant).items()}
print(f"\n{len(CANDIDATES)} candidates: "
      f"{len(VARIANTS)} score-matrix variants x {len(CANDIDATES) // len(VARIANTS)} models")

BioCLIP crop embeddings loaded from notebook 11

10 candidates: 2 score-matrix variants x 5 models


## 8. Selection, on the held-out cameras only

This is what the first run got wrong. Every candidate is scored on the 36 selection cameras —
cameras the heads were never fitted on and that `ood_val` knows nothing about — and the winner
is chosen there. `ood_val` is not read until section 9.

Selection is by **burst** honest macro F1, because that is what gets reported.

In [9]:
SELECTION = {}
for name, fn in CANDIDATES.items():
    probs = fn("train", SEL_MASK)
    per_image = honest_macro_f1(y_sel, probs.argmax(axis=1))
    per_burst = honest_macro_f1(y_sel, aggregate(probs, SEL_FRAME))
    SELECTION[name] = (per_image, per_burst)

print(f"{'candidate':>34} {'sel honest':>11} {'sel burst honest':>18}")
for name in sorted(SELECTION, key=lambda k: -SELECTION[k][1]):
    print(f"{name:>34} {SELECTION[name][0]:>11.3f} {SELECTION[name][1]:>18.3f}")

BEST = max(SELECTION, key=lambda k: SELECTION[k][1])
print(f"\nselected on held-out cameras: {BEST}")

best_variant = BEST.split("[")[-1].rstrip("]")
plain = [k for k in SELECTION if "plain" in k]
spread = [k for k in SELECTION if "spread" in k]
print(f"best plain variant:  {max(plain, key=lambda k: SELECTION[k][1]):<34} "
      f"{max(SELECTION[k][1] for k in plain):.3f}")
print(f"best spread variant: {max(spread, key=lambda k: SELECTION[k][1]):<34} "
      f"{max(SELECTION[k][1] for k in spread):.3f}")

                         candidate  sel honest   sel burst honest
      probe on SpeciesNet  [plain]       0.792              0.954
probe on SpeciesNet  [spread unmapped]       0.793              0.954
               everything  [plain]       0.808              0.951
     everything  [spread unmapped]       0.808              0.948
              both probes  [plain]       0.759              0.918
    both probes  [spread unmapped]       0.757              0.918
                zero shot  [plain]       0.791              0.884
      zero shot  [spread unmapped]       0.788              0.838
         probe on BioCLIP  [plain]       0.667              0.821
probe on BioCLIP  [spread unmapped]       0.667              0.821

selected on held-out cameras: probe on SpeciesNet  [plain]
best plain variant:  probe on SpeciesNet  [plain]       0.954
best spread variant: probe on SpeciesNet  [spread unmapped] 0.954


## 9. The official unseen-camera split

In [10]:
truth = LABELS["ood_val"]
probs = CANDIDATES[BEST]("ood_val")
preds = probs.argmax(axis=1)
seq = aggregate(probs, SPLITS["ood_val"])

print(f"{'view':>12} {'accuracy':>10} {'macro F1':>10} {'honest':>8}")
print(f"{'per image':>12} {(preds == truth).mean():>10.3f} "
      f"{f1_score(truth, preds, average='macro'):>10.3f} "
      f"{honest_macro_f1(truth, preds):>8.3f}")
print(f"{'per burst':>12} {(seq == truth).mean():>10.3f} "
      f"{f1_score(truth, seq, average='macro'):>10.3f} "
      f"{honest_macro_f1(truth, seq):>8.3f}")

print(f"\n{'every candidate, for the record':>40}")
print(f"{'candidate':>34} {'sel burst':>10} {'ood burst honest':>18}")
for name in sorted(CANDIDATES, key=lambda k: -SELECTION[k][1]):
    ood = honest_macro_f1(truth, aggregate(CANDIDATES[name]("ood_val"), SPLITS["ood_val"]))
    star = "  <- selected" if name == BEST else ""
    print(f"{name:>34} {SELECTION[name][1]:>10.3f} {ood:>18.3f}{star}")

import itertools
pairs = [(a, b) for a, b in itertools.combinations(CANDIDATES, 2)
         if SELECTION[a][1] != SELECTION[b][1]]
ood_scores = {k: honest_macro_f1(truth, aggregate(CANDIDATES[k]("ood_val"),
                                                  SPLITS["ood_val"])) for k in CANDIDATES}
agree = sum(1 for a, b in pairs
            if (SELECTION[a][1] - SELECTION[b][1]) * (ood_scores[a] - ood_scores[b]) > 0)
print(f"\nrank agreement between selection and ood_val: "
      f"{agree}/{len(pairs)} ({agree / max(len(pairs), 1):.0%})")
print(f"selection cost {ood_scores[BEST] - max(ood_scores.values()):+.3f} against the")
print("best candidate on ood_val - which could only be known by cheating")

        view   accuracy   macro F1   honest
   per image      0.752      0.734    0.734
   per burst      0.903      0.901    0.908

         every candidate, for the record
                         candidate  sel burst   ood burst honest
      probe on SpeciesNet  [plain]      0.954              0.908  <- selected
probe on SpeciesNet  [spread unmapped]      0.954              0.910
               everything  [plain]      0.951              0.922
     everything  [spread unmapped]      0.948              0.920
              both probes  [plain]      0.918              0.882
    both probes  [spread unmapped]      0.918              0.882
                zero shot  [plain]      0.884              0.825
      zero shot  [spread unmapped]      0.838              0.776
         probe on BioCLIP  [plain]      0.821              0.781
probe on BioCLIP  [spread unmapped]      0.821              0.781

rank agreement between selection and ood_val: 36/43 (84%)
selection cost -0.014 against the


In [11]:
print(classification_report(truth, preds, target_names=CLASS_NAMES, digits=3,
                            zero_division=0))

print(f"\n{'approach':<48} {'accuracy':>10} {'macro F1':>10} {'honest':>8}")
print(f"{'ResNet-18 fine-tuned (v3)':<48} {0.637:>10.3f} {0.543:>10.3f} {0.446:>8.3f}")
print(f"{'EffNetV2-S + XGBoost (nb 07)':<48} {0.581:>10.3f} {0.512:>10.3f} {0.437:>8.3f}")
print(f"{'BioCLIP, full frame (nb 08)':<48} {0.653:>10.3f} {0.581:>10.3f} {0.490:>8.3f}")
print(f"{'BioCLIP + MegaDetector crop (nb 11)':<48} {0.725:>10.3f} {0.662:>10.3f} {0.598:>8.3f}")
print(f"{'   ...per burst':<48} {0.810:>10.3f} {0.763:>10.3f} {0.716:>8.3f}")
print()
print(f"{'SpeciesNet: ' + BEST:<48} {(preds == truth).mean():>10.3f} "
      f"{f1_score(truth, preds, average='macro'):>10.3f} "
      f"{honest_macro_f1(truth, preds):>8.3f}")
print(f"{'   ...per burst':<48} {(seq == truth).mean():>10.3f} "
      f"{f1_score(truth, seq, average='macro'):>10.3f} "
      f"{honest_macro_f1(truth, seq):>8.3f}")

                       precision    recall  f1-score   support

empty (false trigger)      0.409     0.983     0.578       600
     ocellated turkey      0.982     0.737     0.842       600
       great curassow      0.983     0.890     0.934       600
 white-lipped peccary      0.901     0.772     0.831       600
 black-fronted duiker      0.996     0.815     0.896       600
               agouti      0.813     0.414     0.549       210
               ocelot      0.980     0.628     0.765       470
     red brocket deer      0.810     0.623     0.704       130
                 puma      0.623     0.598     0.610       241
             gray fox      0.857     0.503     0.634       310

             accuracy                          0.752      4361
            macro avg      0.836     0.696     0.734      4361
         weighted avg      0.852     0.752     0.770      4361


approach                                           accuracy   macro F1   honest
ResNet-18 fine-tuned (v3)         

## 10. How to read this

**The selected model is the one to quote**, not the best row in section 9's table. Those are
printed for the record; picking among them by their `ood_val` column is what the first run did
wrong. The rank-agreement line says whether the selection cameras were tracking the real task
at all — under 70% and the choice was made on noise.

**Section 6 decides what the `empty` story is.** If "images with NO label mapping" is close to
the 23% of animals that were called empty, the mapping artifact was the cause and the `spread`
variant should win section 8. If instead "animals whose top label is blank" is the large
number, SpeciesNet is genuinely calling these crops empty, and the fix is better crops — the
~18% of images where MegaDetector had no confident box are being handed a whole frame, which
is the wrong input for a crop classifier.

**Zero shot is the headline finding, whatever wins.** It uses none of the 25,211 training
images and still beat notebook 11's fitted 0.598. For a new deployment the first move is to
download SpeciesNet, not to fit anything.

**State the training-data caveat.** SpeciesNet's 65M images include public repositories, and
iWildCam is on LILA. These cameras are unseen to our classifier; they may not be unseen to
SpeciesNet. Notebook 11's 0.598 remains the clean number for "what can be learned from this
training set alone", and this notebook's number is "what is achievable with the best available
camera-trap model".

**What is left.** The ~18% of images with no confident detection are the clearest remaining
loss: they get a whole frame, SpeciesNet calls many of them blank, and their true classes lose
recall. SpeciesNet ships a whole-image classifier variant intended for exactly that case.